In [ ]:

farm_data = {
    # Farm Information
    "farmName": "Demo Farm",
    "location": "Damanhur, Beheira, Egypt",
    "latitude": 31.04,
    "longitude": 30.47,
    "cultivatedArea": 5.0,
    "areaUnit": "Acres",

    # Soil Information
    "soilType": "Sandy loam",
    "soilPH": 7.5,
    "soilSalinityEC": 1.2,       # dS/m
    "soilOrganicMatter": 1.8,    # %

    # Crop Information
    "currentCrop": "Maize",
    "previousWinterCrop": "Wheat",
    "cropHistory": [
        {"crop": "Wheat", "season": "Winter", "year": 2024},
        {"crop": "Maize", "season": "Summer", "year": 2024},
        {"crop": "Wheat", "season": "Winter", "year": 2025},
        {"crop": "Maize", "season": "Summer", "year": 2025}
    ],
    "plantingDate": "2026-05-01",
    "expectedHarvestDate": "2026-09-15",

    # Irrigation Information
    "waterSource": "Nile irrigation canal",
    "irrigationMethod": "Surface irrigation",
    "irrigationFrequency": "Every 10 days",
    "availableWaterAmount": 3000,  # m3 per acre per season

    # Farmer Goal
    "farmerGoal": [
        "Improve soil health",
        "Maintain crop productivity",
        "Reduce water consumption"
    ],
    "goalPriority": [
        "Improve soil health",
        "Maintain crop productivity",
        "Reduce water consumption"
    ]
}


In [21]:
import requests
import pandas as pd
import json

NASA_POWER_URL = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
)
def get_nasa_data(latitude, longitude, start_date, end_date):

    params = {
        "parameters": (
            "T2M,T2M_MAX,T2M_MIN,"
            "PRECTOTCORR,ALLSKY_SFC_SW_DWN"
        ),
        "community": "AG",
        "longitude": longitude,
        "latitude": latitude,
        "start": start_date,
        "end": end_date,
        "format": "JSON",
        "time-standard": "UTC"
    }

    response = requests.get(
        NASA_POWER_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    result = response.json()

    parameter_data = result["properties"]["parameter"]

    df = pd.DataFrame(parameter_data)

    # Convert date index into a normal column
    df.index = pd.to_datetime(
        df.index,
        format="%Y%m%d"
    )

    df.index.name = "date"

    df = df.reset_index()

    # Replace missing values
    df = df.replace(-999, float("nan"))

    return df




In [5]:
latitude = farm_data["latitude"]
longitude = farm_data["longitude"]

df = get_nasa_data(
    latitude=latitude,
    longitude=longitude,
    start_date="20261101",
    end_date="20261114"
)

display(df.head())

,date,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN
0,2026-11-01,23.59,31.01,17.07,0.00,NaN
1,2026-11-02,24.86,34.31,17.79,0.00,NaN
2,2026-11-03,24.24,32.02,19.08,0.02,NaN
3,2026-11-04,23.99,32.57,17.42,0.00,NaN
4,2026-11-05,22.45,28.35,18.66,0.40,NaN


In [6]:
print("Number of observations:", len(df))

display(df.isna().sum())

Number of observations: 14


date                  0
T2M                   2
T2M_MAX               2
T2M_MIN               2
PRECTOTCORR           2
ALLSKY_SFC_SW_DWN    14
dtype: int64

In [ ]:
def safe_round(value, digits=2):

    if pd.isna(value):
        return None

    return round(float(value), digits)



In [8]:
def analyze_environment(df):

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.month

    monthly_temperature = (
        df.groupby("month")["T2M"]
        .mean()
        .reindex(range(1, 13))
    )

    monthly_precipitation = (
        df.groupby("month")["PRECTOTCORR"]
        .sum(min_count=1)
        .reindex(range(1, 13))
    )

    environment_data = {
        "temperature": safe_round(df["T2M"].mean()),

        "precipitation": safe_round(
            df["PRECTOTCORR"].sum(min_count=1)
        ),

        "soilMoisture": None,
        "landSurfaceTemperature": None,
        "vegetationIndex": None,
        "solarRadiation": None,
        "growingSeasonDays": None,

        "monthlyMeanTemperatureC": [
            safe_round(v) for v in monthly_temperature
        ],

        "monthlyPrecipitationMm": [
            safe_round(v) for v in monthly_precipitation
        ]
    }

    return {
        "location": farm_data["location"],

        "period": {
            "start": str(df["date"].min().date()),
            "end": str(df["date"].max().date())
        },

        "averageTemperatureC": safe_round(
            df["T2M"].mean()
        ),

        "maximumDailyTemperatureC": safe_round(
            df["T2M_MAX"].max()
        ),

        "minimumDailyTemperatureC": safe_round(
            df["T2M_MIN"].min()
        ),

        "totalPrecipitationMm": safe_round(
            df["PRECTOTCORR"].sum(min_count=1)
        ),

        "environmentData": environment_data,

        "dataSource": "NASA POWER"
    }

In [9]:
print("DataFrame columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

DataFrame columns:
['date', 'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'ALLSKY_SFC_SW_DWN']

First 5 rows:


,date,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN
0,2026-11-01,23.59,31.01,17.07,0.00,NaN
1,2026-11-02,24.86,34.31,17.79,0.00,NaN
2,2026-11-03,24.24,32.02,19.08,0.02,NaN
3,2026-11-04,23.99,32.57,17.42,0.00,NaN
4,2026-11-05,22.45,28.35,18.66,0.40,NaN



Data types:
date                 datetime64[us]
T2M                         float64
T2M_MAX                     float64
T2M_MIN                     float64
PRECTOTCORR                 float64
ALLSKY_SFC_SW_DWN           float64
dtype: object


In [10]:
environment_summary = analyze_environment(df)

print(
    json.dumps(
        environment_summary,
        indent=4,
        ensure_ascii=False
    )
)

{
    "location": "Damanhur, Beheira, Egypt",
    "period": {
        "start": "2026-11-01",
        "end": "2026-11-14"
    },
    "averageTemperatureC": 21.65,
    "maximumDailyTemperatureC": 34.31,
    "minimumDailyTemperatureC": 13.74,
    "totalPrecipitationMm": 2.8,
    "environmentData": {
        "temperature": 21.65,
        "precipitation": 2.8,
        "soilMoisture": null,
        "landSurfaceTemperature": null,
        "vegetationIndex": null,
        "solarRadiation": null,
        "growingSeasonDays": null,
        "monthlyMeanTemperatureC": [
            null,
            null,
            null,
            null,
            null,
            null,
            null,
            null,
            null,
            null,
            21.65,
            null
        ],
        "monthlyPrecipitationMm": [
            null,
            null,
            null,
            null,
            null,
            null,
            null,
            null,
            null,
          

In [32]:
from langchain_core.tools import tool
from langchain_tavily import TavilySearch


@tool
def get_farm_environment() -> str:
    """
    Retrieve NASA POWER climate data for the demo farm.
    Analyze average temperature, maximum and minimum temperature,
    precipitation, and monthly climate summaries.
    """

    global latest_environment

    df = get_nasa_data(
        latitude=farm_data["latitude"],
        longitude=farm_data["longitude"],
        start_date="20250101",
        end_date="20251231"
    )

    latest_environment = analyze_environment(
        df,
        location=farm_data["location"]
    )

    return json.dumps(
        latest_environment,
        ensure_ascii=False
    )


# Tavily Search Tool
tavily_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="basic",
    include_raw_content=True
)


# All available tools
tools = [
    get_farm_environment,
    tavily_search
]

In [33]:
tavily_result = tavily_search.invoke({
    "query": """
    Scientific studies about crop rotation between wheat,
    Egyptian berseem, faba bean and maize in Egypt.
    Prefer FAO, Egyptian Agricultural Research Center,
    Egyptian universities, and peer-reviewed papers.
    """
})

print(json.dumps(
    tavily_result,
    indent=2,
    ensure_ascii=False
))

{
  "query": "Scientific studies about crop rotation between wheat,\n    Egyptian berseem, faba bean and maize in Egypt.\n    Prefer FAO, Egyptian Agricultural Research Center,\n    Egyptian universities, and peer-reviewed papers.",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.icid.org/ws_inno_mgmt_2015.pdf",
      "title": "[PDF] Use of Crop Rotation to Reduce Climate Change Risks on Major ...",
      "score": 0.60987276,
      "content": "Wheat Peanut/sesame Sugar beet/faba bean Maize/potato Sugar beet/faba bean Sunflower/soybean Clover Maize/cowpea forage Wheat Peanut/sesame Wheat Sesame/peanut Sugar beet/faba bean Maize/potato Clover Maize/cowpea forage Figure (6): Proposed crop rotation in sandy soil in Egypt In sandy soil, the crops grown in both rotations; prevailed and proposed were irrigated with sprinkler or drip system. The proposed rotation saved low amount of water, i.e. 53, 67 and 152 m3/ha in Lower, Middl

In [29]:
system_prompt = """
You are Field Shift, an agricultural decision-support AI agent.

## Objective
Generate exactly 3 distinct crop rotation strategies tailored to the farmer's profile, NASA environmental data, and reliable agricultural research.

## Workflow
1. Analyze the farm profile and crop history.
2. Use NASA tools to retrieve and analyze environmental data for the farm location.
3. Use TavilySearch to find reliable agricultural research on crop requirements, seasonal suitability, water needs, and crop rotation practices.
4. Generate 3 distinct strategies: the current rotation as a baseline (if available) and two evidence-based alternatives.
5. Evaluate and compare strategies based on climate, soil, water availability, crop history, and farmer goals.

## Rules
- Generate crop sequences dynamically; never use hardcoded recommendations.
- Prefer NASA, FAO, Egyptian agricultural research institutions, universities, and peer-reviewed scientific papers.
- Support every important agricultural claim with retrieved evidence.
- Include the actual source title, organization, publication year (if available), and direct URL.
- Never invent data, research, citations, or crop requirements.
- Distinguish NASA observations, published research, and AI interpretations.
- Identify missing information and explain its impact.
- Never claim a strategy is optimal without sufficient evidence.
- If evidence is insufficient, lower confidence and clearly state the limitations.

## Output
Return valid JSON only:

{
  "strategies": [
    {
      "strategy_id": "strategy_1",
      "name": "Strategy name",
      "crop_sequence": ["Winter crop", "Summer crop"],
      "rationale": ["Evidence-based reasons"],
      "potential_benefits": ["Potential benefits"],
      "risks": ["Potential risks"],
      "environmental_analysis": ["NASA findings with values and units"],
      "evidence_sources": [
        {
          "title": "Source title",
          "organization": "Publisher or institution",
          "year": null,
          "url": "Direct source URL",
          "supports": "Claim supported by this source"
        }
      ],
      "missing_information": ["Unavailable data"],
      "confidence_level": "Low | Medium | High"
    }
  ],
  "overall_comparison": "Compare the three strategies and their trade-offs.",
  "recommended_strategy": "Strategy ID or null",
  "recommendation_reason": "Evidence-based explanation"
}

Return exactly 3 strategies. Return only valid JSON.
"""

In [12]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
load_dotenv()

True

In [13]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


memory = InMemorySaver()


agent = create_agent(
    model=llm,

    tools=tools,

    system_prompt=system_prompt,

    checkpointer=memory
)

In [30]:
config = {
    "configurable": {
        "thread_id": "field-shift-beheira-demo"
    }
}

user_message = f"""
I am a farmer in Beheira, Egypt.

Here is my farm profile:

{json.dumps(farm_data, ensure_ascii=False, indent=2)}

Please:

Respond in Arabic.
"""

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_message
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

```json
{
  "strategies": [
    {
      "strategy_id": "strategy_1",
      "name": "الترتيب التقليدي القمح – الذرة",
      "crop_sequence": ["القمح", "الذرة"],
      "rationale": [
        "يستند إلى تاريخ الزراعة الحالي للمنزل (قمح في الشتاء ثم ذرة في الصيف).",
        "بيانات NASA POWER للمنطقة (2025) تُظهر متوسط درجات حرارة 22.73 °C مع إجمالي أمطار 62.51 mm، ما يجعل التبديل بين القمح والذرة مناسباً للظروف المناخية في الدمنهور.",
        "التحول بين حبوب القمح والذرة يقلل من تراكم الآفات ويُحافظ على استقرار الإنتاجية مع نظام الري السطحي المتاح."
      ],
      "potential_benefits": [
        "استمرارية الإنتاجية مع إدارة بسيطة.",
        "تقليل الحاجة إلى تدخلات كيميائية متكررة.",
        "تحسين التوازن الغذائي في التربة عبر التبديل بين حبوب ذات احتياجات مختلفة."
      ],
      "risks": [
        "تراكم نقص العناصر الغذائية، خاصة النيتروجين، مع مرور الوقت.",
        "زيادة خطر تراكم الآفات والمرضى المرتبطين بالحبوب المتكررة."
      ],
      "environmental_analysis": [
        "متوسط 

In [31]:
response

{'messages': [HumanMessage(content='\nI am a farmer in Beheira, Egypt.\n\nHere is my farm profile:\n\n{\n  "farmName": "Demo Farm",\n  "location": "Damanhur, Beheira, Egypt",\n  "latitude": 31.04,\n  "longitude": 30.47,\n  "cultivatedArea": 5.0,\n  "areaUnit": "Acres",\n  "soilType": "Sandy loam",\n  "currentCrop": "Maize",\n  "previousWinterCrop": "Wheat",\n  "waterSource": "Nile irrigation canal",\n  "cropHistory": [\n    "Wheat",\n    "Maize",\n    "Wheat",\n    "Maize"\n  ],\n  "irrigationMethod": "Surface irrigation",\n  "farmerGoal": "Improve soil health and maintain crop productivity"\n}\n\nPlease:\n\nRespond in Arabic.\n', additional_kwargs={}, response_metadata={}, id='c0722216-a565-4322-abad-dc52ea09f905'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to retrieve NASA data for location. Use get_farm_environment. Provide arguments? The function signature: get_farm_environment({_}) => any. Likely expects location coordinates. Provide lat, lon. Let's 

In [24]:
# Check that the strategy tool was executed

if latest_strategies is None:
    raise RuntimeError(
        "The Agent did not generate rotation strategies."
    )

# Verify that we have exactly 3 strategies
assert len(latest_strategies["strategies"]) == 3

# Print structured output
print(
    json.dumps(
        latest_strategies,
        indent=4,
        ensure_ascii=False
    )
)

RuntimeError: The Agent did not generate rotation strategies.

In [33]:
strategies_df = pd.DataFrame(
    [
        {
            "Title": item["title"],
            "Crops": " → ".join(item["crops"]),
            "Badge": item["badgeLabel"],
            "Water Demand": item["waterDemandLabel"]
        }
        for item in latest_strategies["strategies"]
    ]
)

display(strategies_df)

,Title,Crops,Badge,Water Demand
0,Current Rotation,Wheat → Maize,Current Practice,Not assessed
1,Berseem–Maize Rotation,Egyptian Berseem → Maize,Legume Alternative,Not assessed
2,Faba Bean–Maize Rotation,Faba Bean → Maize,Legume Alternative,Not assessed
